In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
data = pd.read_csv("../data/bank_transactions.csv")
data.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance
0,TX000001,AC00128,14.09,4/11/2023 16:29,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21
1,TX000002,AC00455,376.24,6/27/2023 16:44,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91
2,TX000003,AC00019,126.29,7/10/2023 18:16,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35
3,TX000004,AC00070,184.50,5/5/2023 16:32,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06
4,TX000005,AC00411,13.45,10/16/2023 17:51,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40


In [3]:
cols_to_drop = [
    "TransactionID",
    "AccountID",
    "DeviceID",
    "IP Address",
    "MerchantID",
    "TransactionDate"
]

df = data.drop(columns=cols_to_drop)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import IsolationForest

num_features = [
    "TransactionAmount",
    "TransactionDuration",
    "LoginAttempts",
    "AccountBalance",
    "CustomerAge"
]

cat_features = [
    "TransactionType",
    "Channel",
    "Location",
    "CustomerOccupation"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

In [5]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("iso_forest", IsolationForest(
        n_estimators=100,
        contamination=0.02,  # ~2% anomalies
        random_state=42
    ))
])

In [6]:
pipeline.fit(df)

labels = pipeline.predict(df)
df["anomaly"] = labels

In [8]:
df[df["anomaly"] == -1]

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,anomaly
27,40.32,Debit,Milwaukee,Branch,37,Engineer,95,1,2686.52,-1
77,91.53,Debit,Milwaukee,Branch,44,Doctor,69,1,14676.05,-1
90,716.93,Credit,Milwaukee,Online,60,Retired,34,1,4064.02,-1
131,302.45,Credit,San Antonio,Branch,78,Retired,40,1,5133.61,-1
221,485.51,Credit,Milwaukee,Online,76,Retired,37,1,3074.45,-1
...,...,...,...,...,...,...,...,...,...,...
49906,613.13,Debit,Milwaukee,Branch,44,Engineer,28,1,3143.17,-1
49920,369.71,Credit,Milwaukee,Online,72,Retired,71,3,4972.92,-1
49977,259.55,Credit,San Diego,Online,37,Doctor,274,5,12841.01,-1
49978,56.00,Credit,Milwaukee,ATM,56,Retired,161,1,6736.30,-1


In [9]:
df[df["anomaly"] == -1][[
    "TransactionAmount",
    "LoginAttempts",
    "TransactionDuration"
]]

,TransactionAmount,LoginAttempts,TransactionDuration
27,40.32,1,95
77,91.53,1,69
90,716.93,1,34
131,302.45,1,40
221,485.51,1,37
...,...,...,...
49906,613.13,1,28
49920,369.71,3,71
49977,259.55,5,274
49978,56.00,1,161
